In [2]:
import pandas as pd

from PipelineFunctions import (
    amend_bdate,
    generate_customers,
    update_cust_az,
    update_loc_a101,   # NEW
    expand_sales_data,
    generate_sales_for_customers
)

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
PATHS = {
    "cust_az_input": r'C:\Users\neilm\Downloads\CUST_AZ12.csv',
    "cust_info_input": r'C:\Users\neilm\Downloads\Original_cust_info.csv',
    "sales_input": r'C:\Users\neilm\Downloads\Original_sales_details.csv',
    "loc_input": r'C:\Users\neilm\Downloads\LOC_A101.csv',   

    "cust_az_output": r'C:\SQL2025\final_cust_az_amended.csv',
    "cust_info_output": r'C:\SQL2025\final_cust_info.csv',
    "cust_az_updated": r'C:\SQL2025\updated_cust_az.csv',
    "loc_updated": r'C:\SQL2025\updated_LOC_A101.csv',       
    "sales_expanded": r'C:\SQL2025\expanded_sales_data.csv',
    "sales_final": r'C:\SQL2025\Final_sales_data.csv'
}

RANDOM_STATE = 42
N_NEW_CUSTOMERS = 200   # 🔥 CENTRAL CONTROL


# --------------------------------------------------
# STEP 1: AMEND CUSTOMER DOB
# --------------------------------------------------
def step_1_amend_bdate():
    print("Step 1: Amending customer birthdates...")

    cust_az_existing = pd.read_csv(PATHS["cust_az_input"])
    cust_az_amended = amend_bdate(cust_az_existing, RANDOM_STATE)

    cust_az_amended.to_csv(PATHS["cust_az_output"], index=False)

    print("Step 1 complete")
    return cust_az_amended


# --------------------------------------------------
# STEP 2: GENERATE NEW CUSTOMERS
# --------------------------------------------------
def step_2_generate_customers():
    print("Step 2: Generating new customers...")

    df_existing = pd.read_csv(PATHS["cust_info_input"])
    new_customers = generate_customers(df_existing, N_NEW_CUSTOMERS, RANDOM_STATE)

    final_cust_info = pd.concat([df_existing, new_customers], ignore_index=True)
    final_cust_info.to_csv(PATHS["cust_info_output"], index=False)

    print("Step 2 complete")
    return new_customers, final_cust_info


# --------------------------------------------------
# STEP 3: UPDATE CUST_AZ
# --------------------------------------------------
def step_3_update_cust_az(new_customers, cust_az_amended):
    print("Step 3: Updating CUST_AZ...")

    updated_cust_az = update_cust_az(
        new_customers,
        cust_az_amended,
        N_NEW_CUSTOMERS,
        RANDOM_STATE
    )

    updated_cust_az.to_csv(PATHS["cust_az_updated"], index=False)

    print("Step 3 complete")
    return updated_cust_az


# --------------------------------------------------
# STEP 4: UPDATE LOC_A101 (NEW)
# --------------------------------------------------
def step_4_update_loc(new_customers):
    print("Step 4: Updating LOC_A101...")

    loc_existing = pd.read_csv(PATHS["loc_input"])

    updated_loc = update_loc_a101(
        new_customers,
        loc_existing,
        N_NEW_CUSTOMERS,
        RANDOM_STATE
    )

    updated_loc.to_csv(PATHS["loc_updated"], index=False)

    print("Step 4 complete")
    return updated_loc


# --------------------------------------------------
# STEP 5: EXPAND SALES DATA
# --------------------------------------------------
def step_5_expand_sales():
    print("Step 5: Expanding sales data...")

    df_sales = pd.read_csv(PATHS["sales_input"])
    expanded_sales = expand_sales_data(df_sales, 600000, RANDOM_STATE)

    expanded_sales.to_csv(PATHS["sales_expanded"], index=False)

    print("Step 5 complete")
    return expanded_sales


# --------------------------------------------------
# STEP 6: GENERATE SALES FOR NEW CUSTOMERS
# --------------------------------------------------
def step_6_generate_sales(final_cust_info, expanded_sales):
    print("Step 6: Generating sales for new customers...")

    new_sales = generate_sales_for_customers(
        final_cust_info,
        expanded_sales,
        RANDOM_STATE
    )

    final_sales = pd.concat([expanded_sales, new_sales], ignore_index=True)
    final_sales.to_csv(PATHS["sales_final"], index=False)

    print("Step 6 complete")
    return final_sales


# --------------------------------------------------
# MAIN PIPELINE
# --------------------------------------------------
def run_pipeline():
    print("Starting pipeline...\n")

    cust_az_amended = step_1_amend_bdate()
    new_customers, final_cust_info = step_2_generate_customers()
    updated_cust_az = step_3_update_cust_az(new_customers, cust_az_amended)
    updated_loc = step_4_update_loc(new_customers)   
    expanded_sales = step_5_expand_sales()
    final_sales = step_6_generate_sales(final_cust_info, expanded_sales)

    print("\nPipeline completed successfully!")


# --------------------------------------------------
# EXECUTE
# --------------------------------------------------
if __name__ == "__main__":
    run_pipeline()

Starting pipeline...

Step 1: Amending customer birthdates...
Step 1 complete
Step 2: Generating new customers...
Step 2 complete
Step 3: Updating CUST_AZ...
Step 3 complete
Step 4: Updating LOC_A101...
Step 4 complete
Step 5: Expanding sales data...
Step 5 complete
Step 6: Generating sales for new customers...
Step 6 complete

Pipeline completed successfully!
